# Level 6 — Inference Profiling: TinyLlama on T4

This lab profiles **LLM inference** on a Tesla T4 using TinyLlama-1.1B.
We separate and measure the two phases every autoregressive model goes through:

1. **Prefill** — process the entire prompt in parallel (compute-bound)
2. **Decode** — generate tokens one at a time using the KV cache (memory-bandwidth-bound)

We measure:
- **TTFT** (Time to First Token) — how long prefill takes
- **TBT** (Time Between Tokens) — per-token decode latency
- **Throughput** — tokens/sec at different generation lengths
- **KV cache memory** — how GPU memory scales with sequence length

**Hardware:** NVIDIA Tesla T4 (16 GB HBM2, 320 GB/s bandwidth)  
**Model:** TinyLlama/TinyLlama-1.1B-Chat-v1.0 (1.1B params, 22 layers, 32 heads, d=2048)  
**Environment:** Kaggle (T4 GPU) or Google Colab

## 0. Setup

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
import time
import gc
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()

# Model architecture summary
num_params = sum(p.numel() for p in model.parameters())
print(f"\nParameters: {num_params / 1e9:.2f}B")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Heads: {model.config.num_attention_heads}")
print(f"Hidden dim: {model.config.hidden_size}")
print(f"Intermediate dim: {model.config.intermediate_size}")
print(f"Vocab size: {model.config.vocab_size}")
print(f"Max position: {model.config.max_position_embeddings}")

## 1. Baseline memory — model weights only

In [ ]:
torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

baseline_mem = torch.cuda.memory_allocated() / 1e6
print(f"Model weights memory: {baseline_mem:.1f} MB")
print(f"Expected (1.1B params x 2 bytes FP16): {num_params * 2 / 1e6:.1f} MB")

## 2. Prefill vs Decode — manual separation

We manually separate prefill and decode to measure each phase independently.
This avoids `model.generate()` overhead and gives clean per-phase timings.

In [ ]:
def measure_prefill(model, input_ids, n_warmup=3, n_runs=10):
    """Measure prefill latency (forward pass on full prompt, building KV cache)."""
    # Warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = model(input_ids, use_cache=True)
    torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(input_ids, use_cache=True)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    return {
        "mean_ms": np.mean(times) * 1000,
        "std_ms": np.std(times) * 1000,
        "times_ms": [t * 1000 for t in times],
        "past_key_values": outputs.past_key_values,
        "logits": outputs.logits,
    }


def measure_decode_step(model, input_id, past_key_values, n_warmup=3, n_runs=20):
    """Measure a single decode step (one token, reading KV cache)."""
    # Warmup
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = model(input_id, past_key_values=past_key_values, use_cache=True)
    torch.cuda.synchronize()

    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(input_id, past_key_values=past_key_values, use_cache=True)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    return {
        "mean_ms": np.mean(times) * 1000,
        "std_ms": np.std(times) * 1000,
        "times_ms": [t * 1000 for t in times],
    }

In [ ]:
# Test prompt
prompt = "Explain the concept of GPU memory bandwidth in three sentences."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
prompt_len = input_ids.shape[1]
print(f"Prompt: {prompt_len} tokens")

# Measure prefill
prefill_result = measure_prefill(model, input_ids)
print(f"\nPrefill (TTFT): {prefill_result['mean_ms']:.2f} +/- {prefill_result['std_ms']:.2f} ms")
print(f"  Tokens/sec (prefill): {prompt_len / (prefill_result['mean_ms'] / 1000):.0f}")

# Get first generated token for decode measurement
next_token = prefill_result["logits"][:, -1:, :].argmax(dim=-1)
print(f"  First generated token: '{tokenizer.decode(next_token[0])}'")

# Measure single decode step
decode_result = measure_decode_step(model, next_token, prefill_result["past_key_values"])
print(f"\nDecode (TBT): {decode_result['mean_ms']:.2f} +/- {decode_result['std_ms']:.2f} ms")
print(f"  Tokens/sec (decode): {1 / (decode_result['mean_ms'] / 1000):.0f}")

print(f"\nPrefill/Decode ratio: {prefill_result['mean_ms'] / decode_result['mean_ms']:.1f}x")

## 3. Prefill scaling — latency vs prompt length

Prefill processes all tokens in parallel. We expect near-linear scaling
until we saturate compute, then sub-linear (GPU is fully utilized).

In [ ]:
# Generate prompts of varying lengths by repeating tokens
base_text = "The quick brown fox jumps over the lazy dog. " * 50
base_ids = tokenizer(base_text, return_tensors="pt").input_ids.cuda()

prompt_lengths = [16, 32, 64, 128, 256, 512]
prefill_results = []

for seq_len in prompt_lengths:
    ids = base_ids[:, :seq_len]
    result = measure_prefill(model, ids, n_warmup=2, n_runs=5)
    prefill_results.append({
        "seq_len": seq_len,
        "mean_ms": result["mean_ms"],
        "std_ms": result["std_ms"],
        "tokens_per_sec": seq_len / (result["mean_ms"] / 1000),
    })
    print(f"  seq_len={seq_len:>4d}: {result['mean_ms']:>8.2f} ms  "
          f"({seq_len / (result['mean_ms'] / 1000):>8.0f} tok/s)")
    torch.cuda.empty_cache()

print("\nPrefill scaling complete.")

## 4. Decode latency vs KV cache length

Each decode step reads the entire KV cache. As the cache grows, decode
latency should increase — the attention kernel reads more KV entries.

In [ ]:
def measure_decode_at_position(model, tokenizer, context_len, n_runs=20):
    """Build a KV cache of `context_len` tokens, then measure one decode step."""
    base_text = "The quick brown fox jumps over the lazy dog. " * 100
    ids = tokenizer(base_text, return_tensors="pt").input_ids[:, :context_len].cuda()

    # Prefill to build the cache
    with torch.no_grad():
        outputs = model(ids, use_cache=True)
    torch.cuda.synchronize()

    next_token = outputs.logits[:, -1:, :].argmax(dim=-1)
    past = outputs.past_key_values

    # Warmup decode
    for _ in range(3):
        with torch.no_grad():
            _ = model(next_token, past_key_values=past, use_cache=True)
    torch.cuda.synchronize()

    # Measure
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            _ = model(next_token, past_key_values=past, use_cache=True)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    return np.mean(times) * 1000, np.std(times) * 1000


cache_lengths = [32, 64, 128, 256, 512, 1024]
decode_vs_cache = []

print("Decode latency vs KV cache length:")
for ctx_len in cache_lengths:
    mean_ms, std_ms = measure_decode_at_position(model, tokenizer, ctx_len)
    decode_vs_cache.append({"cache_len": ctx_len, "mean_ms": mean_ms, "std_ms": std_ms})
    print(f"  cache_len={ctx_len:>5d}: {mean_ms:>7.2f} +/- {std_ms:.2f} ms  "
          f"({1 / (mean_ms / 1000):>6.0f} tok/s)")
    torch.cuda.empty_cache()
    gc.collect()

## 5. KV cache memory scaling

KV cache memory per token per layer = 2 (K+V) x num_kv_heads x head_dim x 2 bytes (FP16).
We measure actual GPU memory at different sequence lengths and compare to theoretical.

In [ ]:
# Theoretical KV cache size calculation
num_layers = model.config.num_hidden_layers
num_kv_heads = getattr(model.config, "num_key_value_heads", model.config.num_attention_heads)
head_dim = model.config.hidden_size // model.config.num_attention_heads
bytes_per_element = 2  # FP16

# Per token: 2 (K+V) x num_layers x num_kv_heads x head_dim x bytes
kv_bytes_per_token = 2 * num_layers * num_kv_heads * head_dim * bytes_per_element
print(f"KV cache per token (theoretical): {kv_bytes_per_token / 1024:.1f} KB")
print(f"  num_layers={num_layers}, num_kv_heads={num_kv_heads}, head_dim={head_dim}")

# Measure actual memory
seq_lengths = [64, 128, 256, 512, 1024]
memory_results = []

base_text = "The quick brown fox jumps over the lazy dog. " * 200
base_ids = tokenizer(base_text, return_tensors="pt").input_ids.cuda()

print(f"\n{'Seq Len':>8} {'Measured MB':>12} {'Theoretical MB':>15} {'Overhead':>10}")
print("-" * 50)

for seq_len in seq_lengths:
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    mem_before = torch.cuda.memory_allocated()
    ids = base_ids[:, :seq_len]

    with torch.no_grad():
        outputs = model(ids, use_cache=True)
    torch.cuda.synchronize()

    mem_after = torch.cuda.memory_allocated()
    actual_mb = (mem_after - mem_before) / 1e6
    theoretical_mb = seq_len * kv_bytes_per_token / 1e6
    overhead = actual_mb - theoretical_mb

    memory_results.append({
        "seq_len": seq_len,
        "actual_mb": actual_mb,
        "theoretical_mb": theoretical_mb,
        "overhead_mb": overhead,
    })
    print(f"{seq_len:>8d} {actual_mb:>12.2f} {theoretical_mb:>15.2f} {overhead:>+10.2f}")

    del outputs
    torch.cuda.empty_cache()

## 6. End-to-end generation profiling with `model.generate()`

Now we profile the full generation pipeline using `torch.profiler`
to capture GPU kernel traces for both prefill and decode.

In [ ]:
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler

prompt = "Explain why GPU memory bandwidth matters for LLM inference."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
max_new_tokens = 64

# Warmup
with torch.no_grad():
    _ = model.generate(input_ids, max_new_tokens=16, do_sample=False)
torch.cuda.synchronize()

# Profile
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    with_stack=False,
    profile_memory=True,
) as prof:
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    torch.cuda.synchronize()

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
gen_tokens = output_ids.shape[1] - input_ids.shape[1]
print(f"Generated {gen_tokens} tokens")
print(f"Output: {generated_text[:200]}...")

In [ ]:
# Export trace for Perfetto
prof.export_chrome_trace("tinyllama_inference.pt.trace.json")
print("Trace exported to tinyllama_inference.pt.trace.json")
print("Open at https://ui.perfetto.dev to visualize.")

# Top GPU kernels
print("\n--- Top 15 CUDA kernels by total GPU time ---")
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=15,
))

In [ ]:
# Kernel breakdown by category
events = prof.key_averages()

def cuda_total(e):
    """Get total CUDA time, handling API differences across PyTorch versions."""
    if hasattr(e, "cuda_time_total"):
        return e.cuda_time_total
    return e.cuda_time * e.count

total_cuda = sum(cuda_total(e) for e in events if cuda_total(e) > 0)

buckets = {
    "Matmul (linear projections)": ["gemm", "gemv", "addmm"],
    "Attention (SDPA / FMHA)": ["fmha", "flash", "attention", "sdpa", "_efficient"],
    "Softmax": ["softmax"],
    "Elementwise (SiLU, mul, add)": ["silu", "mul", "add", "gelu", "elementwise"],
    "LayerNorm / RMSNorm": ["norm", "layer_norm", "rms"],
    "Embedding / gather": ["embedding", "index_select", "gather"],
    "Copy / transpose": ["copy", "transpose", "permute", "contiguous", "cat"],
}

categorized = {cat: 0 for cat in buckets}
other_time = 0

for e in events:
    ct = cuda_total(e)
    if ct <= 0:
        continue
    name_lower = e.key.lower()
    matched = False
    for cat, keywords in buckets.items():
        if any(kw in name_lower for kw in keywords):
            categorized[cat] += ct
            matched = True
            break
    if not matched:
        other_time += ct

print(f"\nTotal CUDA time: {total_cuda / 1000:.2f} ms")
print(f"\n{chr(39)Category{chr(39):<35s} {chr(39)Time (ms){chr(39):>10s} {chr(39)Share{chr(39):>8s}")
print("-" * 55)
for cat, t in sorted(categorized.items(), key=lambda x: -x[1]):
    if t > 0:
        print(f"{cat:<35s} {t/1000:>10.2f} {t/total_cuda*100:>7.1f}%")
print(f"{chr(39)Other{chr(39):<35s} {other_time/1000:>10.2f} {other_time/total_cuda*100:>7.1f}%")


## 7. Throughput vs generation length

In [ ]:
prompt = "Write a detailed explanation of how transformers work."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()

gen_lengths = [16, 32, 64, 128, 256]
throughput_results = []

print(f"Prompt length: {input_ids.shape[1]} tokens")
print(f"\n{'Gen Length':>10} {'Total (ms)':>12} {'Tokens/sec':>12} {'TBT (ms)':>10}")
print("-" * 48)

for n_tokens in gen_lengths:
    # Warmup
    with torch.no_grad():
        _ = model.generate(input_ids, max_new_tokens=8, do_sample=False)
    torch.cuda.synchronize()

    # Measure (average of 3 runs)
    times = []
    for _ in range(3):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=n_tokens, do_sample=False)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    actual_gen = out.shape[1] - input_ids.shape[1]
    mean_s = np.mean(times)
    tok_per_s = actual_gen / mean_s
    tbt = mean_s / actual_gen * 1000

    throughput_results.append({
        "gen_length": actual_gen,
        "total_ms": mean_s * 1000,
        "tokens_per_sec": tok_per_s,
        "tbt_ms": tbt,
    })
    print(f"{actual_gen:>10d} {mean_s * 1000:>12.1f} {tok_per_s:>12.1f} {tbt:>10.2f}")
    torch.cuda.empty_cache()

## 8. Roofline context — why decode is memory-bound

Quick arithmetic to confirm decode is memory-bandwidth-bound on T4.

In [ ]:
# T4 specs
t4_bandwidth_gb_s = 320  # GB/s HBM2
t4_fp16_tflops = 65      # TFLOPS (Tensor Cores)

# Model weights
model_size_gb = num_params * 2 / 1e9  # FP16

# Decode: each token requires loading all model weights once
# (plus KV cache, but weights dominate at short sequences)
weight_load_time_ms = model_size_gb / t4_bandwidth_gb_s * 1000

# FLOPs per token: ~2 * num_params (rough estimate for transformer forward pass)
flops_per_token = 2 * num_params
compute_time_ms = flops_per_token / (t4_fp16_tflops * 1e12) * 1000

# Arithmetic intensity
ai = flops_per_token / (num_params * 2)  # FLOPs / bytes loaded
ridge_point = t4_fp16_tflops * 1e12 / (t4_bandwidth_gb_s * 1e9)  # FLOPs/byte

print("=== Decode roofline analysis (batch=1) ===")
print(f"\nModel size (FP16): {model_size_gb:.2f} GB")
print(f"Weight load time:  {weight_load_time_ms:.2f} ms")
print(f"Compute time:      {compute_time_ms:.4f} ms")
print(f"\nArithmetic intensity: {ai:.1f} FLOPs/byte")
print(f"T4 FP16 ridge point:  {ridge_point:.1f} FLOPs/byte")
print(f"\nVerdict: {'MEMORY-BOUND' if ai < ridge_point else 'COMPUTE-BOUND'}")
print(f"  (AI {ai:.1f} << ridge {ridge_point:.1f})")
print(f"\nTheoretical max decode throughput (batch=1): "
      f"{t4_bandwidth_gb_s / model_size_gb:.0f} tokens/sec")
print(f"Measured decode throughput: "
      f"{1 / (decode_result['mean_ms'] / 1000):.0f} tokens/sec")
print(f"Memory bandwidth utilization: "
      f"{(1 / (decode_result['mean_ms'] / 1000)) / (t4_bandwidth_gb_s / model_size_gb) * 100:.1f}%")

## 9. Summary plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Level 6 — TinyLlama Inference Profiling on T4", fontsize=14, fontweight="bold")

# 1. Prefill latency vs sequence length
ax = axes[0, 0]
seq_lens = [r["seq_len"] for r in prefill_results]
prefill_ms = [r["mean_ms"] for r in prefill_results]
ax.plot(seq_lens, prefill_ms, "o-", color="#2196F3", linewidth=2)
ax.set_xlabel("Prompt length (tokens)")
ax.set_ylabel("Prefill latency (ms)")
ax.set_title("Prefill (TTFT) vs Prompt Length")
ax.grid(True, alpha=0.3)

# 2. Decode latency vs KV cache length
ax = axes[0, 1]
cache_lens = [r["cache_len"] for r in decode_vs_cache]
decode_ms = [r["mean_ms"] for r in decode_vs_cache]
ax.plot(cache_lens, decode_ms, "s-", color="#FF5722", linewidth=2)
ax.set_xlabel("KV cache length (tokens)")
ax.set_ylabel("Decode latency per token (ms)")
ax.set_title("Decode (TBT) vs KV Cache Size")
ax.grid(True, alpha=0.3)

# 3. KV cache memory
ax = axes[1, 0]
mem_seqs = [r["seq_len"] for r in memory_results]
mem_actual = [r["actual_mb"] for r in memory_results]
mem_theory = [r["theoretical_mb"] for r in memory_results]
ax.plot(mem_seqs, mem_actual, "o-", label="Measured", color="#4CAF50", linewidth=2)
ax.plot(mem_seqs, mem_theory, "--", label="Theoretical", color="#9E9E9E", linewidth=2)
ax.set_xlabel("Sequence length (tokens)")
ax.set_ylabel("KV cache memory (MB)")
ax.set_title("KV Cache Memory Scaling")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Throughput vs generation length
ax = axes[1, 1]
gen_lens = [r["gen_length"] for r in throughput_results]
tps = [r["tokens_per_sec"] for r in throughput_results]
ax.plot(gen_lens, tps, "D-", color="#9C27B0", linewidth=2)
ax.set_xlabel("Generation length (tokens)")
ax.set_ylabel("Throughput (tokens/sec)")
ax.set_title("Decode Throughput vs Generation Length")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("inference_profiling_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved to inference_profiling_plots.png")

## 10. Summary

**Fill in after running the notebook with actual measured numbers.**

| Metric | Value |
|---|---|
| Prefill (TTFT), 14-token prompt | ___ ms |
| Decode (TBT), single step | ___ ms |
| Decode throughput (batch=1) | ___ tok/s |
| KV cache per token | ___ KB |
| Memory bandwidth utilization | ___% |
| Decode arithmetic intensity | ___ FLOPs/byte |
| Verdict | Memory-bound (AI << ridge) |

### Key takeaways

1. **Prefill is compute-bound, decode is memory-bound.** Prefill processes all tokens in parallel and saturates compute. Decode generates one token at a time, loading all model weights for a single output — throughput is limited by HBM bandwidth.

2. **KV cache memory scales linearly.** Each token adds a fixed amount of memory (2 x layers x kv_heads x head_dim x 2 bytes). For TinyLlama this is ~___ KB/token. At 2048 tokens the cache alone uses ~___ MB.

3. **Decode TBT increases with sequence length** because the attention kernel must read a larger KV cache. The effect is modest at short sequences but becomes significant as context grows.

4. **Batch=1 decode barely utilizes the GPU.** The arithmetic intensity is ~1 FLOP/byte, far below the T4's FP16 ridge point of ~203. This is why serving systems batch multiple requests — higher batch size raises AI toward the ridge.